# Coconut Leaf Disease Detection Model Training (v2)

This notebook trains a deep learning model for detecting coconut leaf diseases.

## Classes:
1. **healthy** - Healthy coconut leaves
2. **Leaf Rot** - Leaves affected by rot disease
3. **Leaf_Spot** - Leaves with spot disease
4. **not_cocount** - Non-coconut images (rejection class)

## Requirements from Supervisor:
- Check Precision, Recall, F1-score for each class (class-wise)
- Precision, Recall, F1-score should be close to each other
- Similar values for all classes
- Accuracy should be close to F1
- No data leaking
- No overfitting

## 1. Import Libraries and Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

# TensorFlow imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, 
    ModelCheckpoint, 
    ReduceLROnPlateau,
    TensorBoard
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

# Sklearn imports for metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score
)

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Configuration and Paths

In [ ]:
# Configuration
CONFIG = {
    'img_size': 224,
    'batch_size': 32,
    'epochs': 50,
    'learning_rate': 0.0001,
    'dropout_rate': 0.5,
    'l2_reg': 0.01,
    'patience_early_stop': 10,
    'patience_reduce_lr': 5,
}

# Dataset paths
BASE_PATH = r'C:\Users\Tharindu Nandun\Desktop\Research\Research\ml\data\raw\stage_2_split'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
VAL_PATH = os.path.join(BASE_PATH, 'val')
TEST_PATH = os.path.join(BASE_PATH, 'test')

# Model save path
MODEL_VERSION = 'v2'
MODEL_DIR = r'C:\Users\Tharindu Nandun\Desktop\Research\Research\ml\models\leaf_disease_v2'
os.makedirs(MODEL_DIR, exist_ok=True)

# Class names (will be auto-detected)
CLASS_NAMES = sorted(os.listdir(TRAIN_PATH))
NUM_CLASSES = len(CLASS_NAMES)

print(f"Configuration: {json.dumps(CONFIG, indent=2)}")
print(f"\nClasses detected: {CLASS_NAMES}")
print(f"Number of classes: {NUM_CLASSES}")

## 3. Dataset Analysis

In [ ]:
def count_images(path):
    """Count images in each class folder."""
    counts = {}
    for class_name in CLASS_NAMES:
        class_path = os.path.join(path, class_name)
        if os.path.exists(class_path):
            counts[class_name] = len([f for f in os.listdir(class_path) 
                                      if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        else:
            counts[class_name] = 0
    return counts

# Count images in each split
train_counts = count_images(TRAIN_PATH)
val_counts = count_images(VAL_PATH)
test_counts = count_images(TEST_PATH)

# Create summary DataFrame
dataset_summary = pd.DataFrame({
    'Class': CLASS_NAMES,
    'Train': [train_counts[c] for c in CLASS_NAMES],
    'Validation': [val_counts[c] for c in CLASS_NAMES],
    'Test': [test_counts[c] for c in CLASS_NAMES]
})
dataset_summary['Total'] = dataset_summary['Train'] + dataset_summary['Validation'] + dataset_summary['Test']

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(dataset_summary.to_string(index=False))
print("\n" + "=" * 60)
print(f"Total Train: {dataset_summary['Train'].sum()}")
print(f"Total Validation: {dataset_summary['Validation'].sum()}")
print(f"Total Test: {dataset_summary['Test'].sum()}")
print(f"Total Images: {dataset_summary['Total'].sum()}")

In [ ]:
# Visualize dataset distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES))

# Train distribution
axes[0].bar(CLASS_NAMES, [train_counts[c] for c in CLASS_NAMES], color=colors)
axes[0].set_title('Training Set Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Images')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate([train_counts[c] for c in CLASS_NAMES]):
    axes[0].text(i, v + 50, str(v), ha='center', fontsize=9)

# Validation distribution
axes[1].bar(CLASS_NAMES, [val_counts[c] for c in CLASS_NAMES], color=colors)
axes[1].set_title('Validation Set Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Number of Images')
axes[1].tick_params(axis='x', rotation=45)
for i, v in enumerate([val_counts[c] for c in CLASS_NAMES]):
    axes[1].text(i, v + 5, str(v), ha='center', fontsize=9)

# Test distribution
axes[2].bar(CLASS_NAMES, [test_counts[c] for c in CLASS_NAMES], color=colors)
axes[2].set_title('Test Set Distribution', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Class')
axes[2].set_ylabel('Number of Images')
axes[2].tick_params(axis='x', rotation=45)
for i, v in enumerate([test_counts[c] for c in CLASS_NAMES]):
    axes[2].text(i, v + 5, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'dataset_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print("\nDataset distribution chart saved!")

## 4. Data Loading and Augmentation

### Important: No Data Leaking!
- Train, Validation, and Test sets are completely separate
- Data augmentation is ONLY applied to training data
- Validation and Test data use only rescaling (no augmentation)

In [ ]:
# Training data generator WITH augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest'
)

# Validation and Test data generators WITHOUT augmentation (only rescaling)
val_test_datagen = ImageDataGenerator(rescale=1./255)

print("Data generators created:")
print("- Training: WITH augmentation (rotation, shift, flip, zoom)")
print("- Validation: NO augmentation (rescale only)")
print("- Test: NO augmentation (rescale only)")

In [ ]:
# Create data generators
train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_PATH,
    target_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_PATH,
    target_size=(CONFIG['img_size'], CONFIG['img_size']),
    batch_size=CONFIG['batch_size'],
    class_mode='categorical',
    shuffle=False
)

# Store class indices mapping
class_indices = train_generator.class_indices
idx_to_class = {v: k for k, v in class_indices.items()}

print(f"\nClass indices mapping: {class_indices}")
print(f"\nTrain samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Test samples: {test_generator.samples}")

In [ ]:
# Visualize sample augmented images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Training Images (with Augmentation)', fontsize=14, fontweight='bold')

# Get a batch of images
sample_batch = next(train_generator)
images, labels = sample_batch

for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i])
        label_idx = np.argmax(labels[i])
        ax.set_title(f'Class: {idx_to_class[label_idx]}', fontsize=10)
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'sample_augmented_images.png'), dpi=150, bbox_inches='tight')
plt.show()

# Reset generator
train_generator.reset()

## 5. Calculate Class Weights for Imbalanced Data

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Calculate class weights to handle imbalanced data
train_labels = train_generator.classes
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = dict(enumerate(class_weights_array))

print("=" * 50)
print("CLASS WEIGHTS (for handling imbalanced data)")
print("=" * 50)
for idx, weight in class_weights.items():
    print(f"  {idx_to_class[idx]}: {weight:.4f}")
print("\nHigher weight = less samples = model pays more attention")

## 6. Build Model Architecture

Using **EfficientNetB0** as base model with:
- Dropout for regularization (prevent overfitting)
- L2 regularization
- BatchNormalization for stable training

In [ ]:
def build_model(num_classes, img_size=224, dropout_rate=0.5, l2_reg=0.01):
    """Build EfficientNetB0-based classification model with anti-overfitting techniques."""
    
    # Load pre-trained EfficientNetB0 (without top layers)
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(img_size, img_size, 3)
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Build model
    inputs = keras.Input(shape=(img_size, img_size, 3))
    
    # Base model
    x = base_model(inputs, training=False)
    
    # Global average pooling
    x = layers.GlobalAveragePooling2D()(x)
    
    # Dense layers with regularization
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=l2(l2_reg))(x)
    x = layers.Dropout(dropout_rate)(x)
    
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=l2(l2_reg))(x)
    x = layers.Dropout(dropout_rate)(x)
    
    # Output layer
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    
    return model, base_model

# Build the model
model, base_model = build_model(
    num_classes=NUM_CLASSES,
    img_size=CONFIG['img_size'],
    dropout_rate=CONFIG['dropout_rate'],
    l2_reg=CONFIG['l2_reg']
)

print("Model built successfully!")
print(f"\nAnti-overfitting techniques used:")
print(f"  - Dropout rate: {CONFIG['dropout_rate']}")
print(f"  - L2 regularization: {CONFIG['l2_reg']}")
print(f"  - Data augmentation: Yes (training only)")
print(f"  - Early stopping: Yes")
print(f"  - Learning rate reduction: Yes")

In [ ]:
# Model summary
model.summary()

In [ ]:
# Compile model
model.compile(
    optimizer=Adam(learning_rate=CONFIG['learning_rate']),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled with:")
print(f"  - Optimizer: Adam (lr={CONFIG['learning_rate']})")
print(f"  - Loss: categorical_crossentropy")
print(f"  - Metrics: accuracy")

## 7. Setup Callbacks

In [ ]:
# Define callbacks
callbacks = [
    # Early stopping to prevent overfitting
    EarlyStopping(
        monitor='val_loss',
        patience=CONFIG['patience_early_stop'],
        restore_best_weights=True,
        verbose=1
    ),
    
    # Save best model
    ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'best_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    
    # Reduce learning rate when stuck
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=CONFIG['patience_reduce_lr'],
        min_lr=1e-7,
        verbose=1
    )
]

print("Callbacks configured:")
print(f"  - EarlyStopping (patience={CONFIG['patience_early_stop']})")
print(f"  - ModelCheckpoint (save best model)")
print(f"  - ReduceLROnPlateau (patience={CONFIG['patience_reduce_lr']})")

## 8. Phase 1: Train with Frozen Base Model

In [ ]:
print("=" * 60)
print("PHASE 1: Training with frozen base model")
print("=" * 60)
print(f"Base model trainable: {base_model.trainable}")
print(f"Epochs: {CONFIG['epochs'] // 2}")
print("\nStarting training...\n")

# Train Phase 1
history_phase1 = model.fit(
    train_generator,
    epochs=CONFIG['epochs'] // 2,
    validation_data=val_generator,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

print("\nPhase 1 training completed!")

## 9. Phase 2: Fine-tuning with Unfrozen Base Model

In [ ]:
print("=" * 60)
print("PHASE 2: Fine-tuning with unfrozen base model")
print("=" * 60)

# Unfreeze the base model
base_model.trainable = True

# Freeze first 100 layers, fine-tune the rest
for layer in base_model.layers[:100]:
    layer.trainable = False

trainable_layers = sum([1 for layer in base_model.layers if layer.trainable])
print(f"Unfrozen layers in base model: {trainable_layers}")

# Recompile with lower learning rate
model.compile(
    optimizer=Adam(learning_rate=CONFIG['learning_rate'] / 10),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"New learning rate: {CONFIG['learning_rate'] / 10}")
print("\nStarting fine-tuning...\n")

# Train Phase 2
history_phase2 = model.fit(
    train_generator,
    epochs=CONFIG['epochs'] // 2,
    validation_data=val_generator,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

print("\nPhase 2 fine-tuning completed!")

## 10. Training History Visualization

In [ ]:
# Combine training history
def combine_histories(h1, h2):
    """Combine two training histories."""
    combined = {}
    for key in h1.history.keys():
        combined[key] = h1.history[key] + h2.history[key]
    return combined

history = combine_histories(history_phase1, history_phase2)

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0].plot(history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[0].axvline(x=len(history_phase1.history['accuracy'])-1, color='r', linestyle='--', 
                label='Fine-tuning Start', alpha=0.7)
axes[0].set_title('Model Accuracy Over Epochs', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(history['loss'], label='Train Loss', linewidth=2)
axes[1].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[1].axvline(x=len(history_phase1.history['loss'])-1, color='r', linestyle='--', 
                label='Fine-tuning Start', alpha=0.7)
axes[1].set_title('Model Loss Over Epochs', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Training curves saved!")

In [ ]:
# Check for overfitting
final_train_acc = history['accuracy'][-1]
final_val_acc = history['val_accuracy'][-1]
final_train_loss = history['loss'][-1]
final_val_loss = history['val_loss'][-1]

print("=" * 50)
print("OVERFITTING CHECK")
print("=" * 50)
print(f"Final Train Accuracy: {final_train_acc:.4f}")
print(f"Final Val Accuracy:   {final_val_acc:.4f}")
print(f"Accuracy Gap:         {abs(final_train_acc - final_val_acc):.4f}")
print(f"\nFinal Train Loss: {final_train_loss:.4f}")
print(f"Final Val Loss:   {final_val_loss:.4f}")
print(f"Loss Gap:         {abs(final_train_loss - final_val_loss):.4f}")

if abs(final_train_acc - final_val_acc) < 0.1:
    print("\n[OK] No significant overfitting detected!")
else:
    print("\n[WARNING] Possible overfitting - gap > 10%")

## 11. Model Evaluation on Test Set

In [ ]:
# Load best model
best_model_path = os.path.join(MODEL_DIR, 'best_model.keras')
if os.path.exists(best_model_path):
    model = keras.models.load_model(best_model_path)
    print(f"Best model loaded from: {best_model_path}")
else:
    print("Using current model (best model file not found)")

# Evaluate on test set
print("\nEvaluating on test set...")
test_generator.reset()
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

## 12. Generate Predictions

In [ ]:
# Generate predictions on test set
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = test_generator.classes

print(f"\nTotal predictions: {len(predicted_classes)}")
print(f"Total true labels: {len(true_classes)}")

## 13. Class-wise Metrics: Precision, Recall, F1-Score

### Supervisor Requirements:
- Check Precision, Recall, F1-score for each class
- Values should be close to each other
- Similar values across all classes
- Accuracy should be close to F1

In [ ]:
# Calculate class-wise metrics
precision, recall, f1, support = precision_recall_fscore_support(
    true_classes, predicted_classes, average=None
)

# Create metrics DataFrame
class_metrics_df = pd.DataFrame({
    'Class': [idx_to_class[i] for i in range(NUM_CLASSES)],
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support.astype(int)
})

# Add P-R-F1 difference column
class_metrics_df['P-R-F1 Max Diff'] = class_metrics_df.apply(
    lambda row: max(row['Precision'], row['Recall'], row['F1-Score']) - 
                min(row['Precision'], row['Recall'], row['F1-Score']), axis=1
)

print("=" * 70)
print("CLASS-WISE METRICS (Precision, Recall, F1-Score)")
print("=" * 70)
print(class_metrics_df.to_string(index=False, float_format='{:.4f}'.format))

# Save to CSV
class_metrics_df.to_csv(os.path.join(MODEL_DIR, 'class_metrics.csv'), index=False)
print(f"\nClass metrics saved to: {os.path.join(MODEL_DIR, 'class_metrics.csv')}")

In [ ]:
# Calculate overall metrics
accuracy = accuracy_score(true_classes, predicted_classes)

# Macro averages (treats all classes equally)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    true_classes, predicted_classes, average='macro'
)

# Weighted averages (accounts for class imbalance)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    true_classes, predicted_classes, average='weighted'
)

print("=" * 70)
print("OVERALL METRICS SUMMARY")
print("=" * 70)
print(f"\nAccuracy:          {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"\n--- Macro Average (treats all classes equally) ---")
print(f"Precision (macro): {precision_macro:.4f}")
print(f"Recall (macro):    {recall_macro:.4f}")
print(f"F1-Score (macro):  {f1_macro:.4f}")
print(f"\n--- Weighted Average (accounts for class sizes) ---")
print(f"Precision (weighted): {precision_weighted:.4f}")
print(f"Recall (weighted):    {recall_weighted:.4f}")
print(f"F1-Score (weighted):  {f1_weighted:.4f}")

In [ ]:
# Supervisor Check: Accuracy vs F1 comparison
print("=" * 70)
print("SUPERVISOR CHECK: Accuracy vs F1-Score Comparison")
print("=" * 70)

acc_f1_diff_macro = abs(accuracy - f1_macro)
acc_f1_diff_weighted = abs(accuracy - f1_weighted)

print(f"\nAccuracy:            {accuracy:.4f}")
print(f"F1-Score (macro):    {f1_macro:.4f}")
print(f"F1-Score (weighted): {f1_weighted:.4f}")
print(f"\n|Accuracy - F1_macro|:    {acc_f1_diff_macro:.4f}")
print(f"|Accuracy - F1_weighted|: {acc_f1_diff_weighted:.4f}")

if acc_f1_diff_macro < 0.05:
    print("\n[OK] Accuracy is close to F1-Score (macro)!")
else:
    print("\n[INFO] There is some difference between Accuracy and F1-Score")

# Check if P, R, F1 are close for each class
print("\n" + "=" * 70)
print("SUPERVISOR CHECK: P-R-F1 Balance per Class")
print("=" * 70)

for i, row in class_metrics_df.iterrows():
    if row['P-R-F1 Max Diff'] < 0.1:
        status = "[OK]"
    elif row['P-R-F1 Max Diff'] < 0.2:
        status = "[GOOD]"
    else:
        status = "[NEEDS IMPROVEMENT]"
    print(f"{status} {row['Class']}: P={row['Precision']:.3f}, R={row['Recall']:.3f}, F1={row['F1-Score']:.3f}, Diff={row['P-R-F1 Max Diff']:.3f}")

## 14. Classification Report

In [ ]:
# Full classification report
print("=" * 70)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 70)

class_names_list = [idx_to_class[i] for i in range(NUM_CLASSES)]
report = classification_report(true_classes, predicted_classes, target_names=class_names_list)
print(report)

# Save report
with open(os.path.join(MODEL_DIR, 'classification_report.txt'), 'w') as f:
    f.write("CLASSIFICATION REPORT\n")
    f.write("=" * 70 + "\n")
    f.write(report)
    f.write(f"\nTest Accuracy: {accuracy:.4f}")

print(f"\nClassification report saved!")

## 15. Confusion Matrix Visualization

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names_list, yticklabels=class_names_list, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Normalized (percentage)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues', 
            xticklabels=class_names_list, yticklabels=class_names_list, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized %)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted Label')
axes[1].set_ylabel('True Label')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Confusion matrix saved!")

## 16. Class-wise Metrics Bar Chart

In [ ]:
# Bar chart for class-wise metrics
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(NUM_CLASSES)
width = 0.25

bars1 = ax.bar(x - width, class_metrics_df['Precision'], width, label='Precision', color='#2ecc71')
bars2 = ax.bar(x, class_metrics_df['Recall'], width, label='Recall', color='#3498db')
bars3 = ax.bar(x + width, class_metrics_df['F1-Score'], width, label='F1-Score', color='#e74c3c')

# Add value labels on bars
def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom', fontsize=8)

add_labels(bars1)
add_labels(bars2)
add_labels(bars3)

ax.set_xlabel('Class', fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Class-wise Precision, Recall, and F1-Score', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_metrics_df['Class'], rotation=15)
ax.legend()
ax.set_ylim(0, 1.15)
ax.axhline(y=accuracy, color='purple', linestyle='--', label=f'Accuracy ({accuracy:.2f})', alpha=0.7)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'class_metrics_chart.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Class metrics chart saved!")

## 17. Metrics Balance Analysis

In [ ]:
# Analyze metrics balance across classes
print("=" * 70)
print("METRICS BALANCE ANALYSIS")
print("=" * 70)

# Calculate standard deviation across classes (lower = more balanced)
precision_std = np.std(class_metrics_df['Precision'])
recall_std = np.std(class_metrics_df['Recall'])
f1_std = np.std(class_metrics_df['F1-Score'])

print(f"\nStandard Deviation Across Classes:")
print(f"  Precision: {precision_std:.4f} (lower = more balanced)")
print(f"  Recall:    {recall_std:.4f}")
print(f"  F1-Score:  {f1_std:.4f}")

# Calculate min-max range
print(f"\nMin-Max Range Across Classes:")
print(f"  Precision: {class_metrics_df['Precision'].min():.4f} - {class_metrics_df['Precision'].max():.4f} (range: {class_metrics_df['Precision'].max() - class_metrics_df['Precision'].min():.4f})")
print(f"  Recall:    {class_metrics_df['Recall'].min():.4f} - {class_metrics_df['Recall'].max():.4f} (range: {class_metrics_df['Recall'].max() - class_metrics_df['Recall'].min():.4f})")
print(f"  F1-Score:  {class_metrics_df['F1-Score'].min():.4f} - {class_metrics_df['F1-Score'].max():.4f} (range: {class_metrics_df['F1-Score'].max() - class_metrics_df['F1-Score'].min():.4f})")

# Overall balance assessment
avg_std = (precision_std + recall_std + f1_std) / 3
print(f"\n--- Overall Balance Assessment ---")
if avg_std < 0.05:
    print(f"[EXCELLENT] Metrics are very well balanced across classes (avg std: {avg_std:.4f})")
elif avg_std < 0.10:
    print(f"[GOOD] Metrics are reasonably balanced across classes (avg std: {avg_std:.4f})")
elif avg_std < 0.15:
    print(f"[ACCEPTABLE] Some imbalance in metrics across classes (avg std: {avg_std:.4f})")
else:
    print(f"[NEEDS IMPROVEMENT] Significant imbalance in metrics across classes (avg std: {avg_std:.4f})")

In [ ]:
# Radar chart for class-wise metrics
from math import pi

# Prepare data for radar chart
categories = ['Precision', 'Recall', 'F1-Score']
N = len(categories)

# Create figure
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

# Compute angle for each category
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Close the polygon

# Colors for each class
colors = plt.cm.Set2(np.linspace(0, 1, NUM_CLASSES))

# Plot each class
for i, (_, row) in enumerate(class_metrics_df.iterrows()):
    values = [row['Precision'], row['Recall'], row['F1-Score']]
    values += values[:1]  # Close the polygon
    ax.plot(angles, values, 'o-', linewidth=2, label=row['Class'], color=colors[i])
    ax.fill(angles, values, alpha=0.1, color=colors[i])

# Set labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 1)
ax.set_title('Class-wise Metrics Radar Chart', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'metrics_radar_chart.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Radar chart saved!")

## 18. Sample Predictions Visualization

In [ ]:
# Visualize sample predictions
test_generator.reset()
sample_batch = next(test_generator)
sample_images, sample_labels = sample_batch

# Get predictions for this batch
sample_predictions = model.predict(sample_images, verbose=0)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Sample Test Predictions', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i < len(sample_images):
        ax.imshow(sample_images[i])
        true_idx = np.argmax(sample_labels[i])
        pred_idx = np.argmax(sample_predictions[i])
        confidence = sample_predictions[i][pred_idx] * 100
        
        true_label = idx_to_class[true_idx]
        pred_label = idx_to_class[pred_idx]
        
        # Color code: green for correct, red for incorrect
        color = 'green' if true_idx == pred_idx else 'red'
        ax.set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.1f}%)', 
                     fontsize=9, color=color)
        ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Sample predictions visualization saved!")

## 19. Save Model Info and Summary

In [ ]:
# Save model info as JSON
model_info = {
    'model_name': 'Coconut Leaf Disease Detection',
    'version': MODEL_VERSION,
    'created_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'base_model': 'EfficientNetB0',
    'input_shape': [CONFIG['img_size'], CONFIG['img_size'], 3],
    'num_classes': NUM_CLASSES,
    'classes': class_names_list,
    'class_indices': class_indices,
    'training_config': CONFIG,
    'performance': {
        'test_accuracy': float(accuracy),
        'test_loss': float(test_loss),
        'precision_macro': float(precision_macro),
        'recall_macro': float(recall_macro),
        'f1_macro': float(f1_macro),
        'precision_weighted': float(precision_weighted),
        'recall_weighted': float(recall_weighted),
        'f1_weighted': float(f1_weighted)
    },
    'class_metrics': {
        class_name: {
            'precision': float(class_metrics_df[class_metrics_df['Class']==class_name]['Precision'].values[0]),
            'recall': float(class_metrics_df[class_metrics_df['Class']==class_name]['Recall'].values[0]),
            'f1_score': float(class_metrics_df[class_metrics_df['Class']==class_name]['F1-Score'].values[0]),
            'support': int(class_metrics_df[class_metrics_df['Class']==class_name]['Support'].values[0])
        }
        for class_name in class_names_list
    },
    'anti_overfitting_techniques': [
        'Dropout (0.5)',
        'L2 Regularization (0.01)',
        'Data Augmentation (training only)',
        'Early Stopping',
        'Learning Rate Reduction',
        'Class Weights for Imbalanced Data'
    ]
}

with open(os.path.join(MODEL_DIR, 'model_info.json'), 'w') as f:
    json.dump(model_info, f, indent=2)

print("Model info saved!")
print(json.dumps(model_info, indent=2))

## 20. Final Summary Report

In [ ]:
print("="*70)
print("FINAL MODEL TRAINING SUMMARY")
print("="*70)
print(f"\nModel: Coconut Leaf Disease Detection {MODEL_VERSION}")
print(f"Base Architecture: EfficientNetB0")
print(f"Number of Classes: {NUM_CLASSES}")
print(f"Classes: {class_names_list}")

print(f"\n--- Dataset ---")
print(f"Train: {sum(train_counts.values())} images")
print(f"Validation: {sum(val_counts.values())} images")
print(f"Test: {sum(test_counts.values())} images")

print(f"\n--- Test Performance ---")
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1-Score (macro): {f1_macro:.4f}")
print(f"F1-Score (weighted): {f1_weighted:.4f}")
print(f"|Accuracy - F1_macro|: {abs(accuracy - f1_macro):.4f}")

print(f"\n--- Class-wise Metrics ---")
print(class_metrics_df.to_string(index=False))

print(f"\n--- Supervisor Requirements Check ---")
print(f"[{'OK' if abs(accuracy - f1_macro) < 0.05 else 'CHECK'}] Accuracy close to F1: {abs(accuracy - f1_macro):.4f}")
print(f"[{'OK' if avg_std < 0.10 else 'CHECK'}] Balanced metrics across classes: std={avg_std:.4f}")
print(f"[OK] No data leaking (separate train/val/test splits)")
print(f"[{'OK' if abs(final_train_acc - final_val_acc) < 0.1 else 'CHECK'}] No overfitting: gap={abs(final_train_acc - final_val_acc):.4f}")

print(f"\n--- Files Saved ---")
print(f"Model: {os.path.join(MODEL_DIR, 'best_model.keras')}")
print(f"Model Info: {os.path.join(MODEL_DIR, 'model_info.json')}")
print(f"Class Metrics: {os.path.join(MODEL_DIR, 'class_metrics.csv')}")
print(f"Classification Report: {os.path.join(MODEL_DIR, 'classification_report.txt')}")
print(f"\nCharts:")
print(f"  - dataset_distribution.png")
print(f"  - training_curves.png")
print(f"  - confusion_matrix.png")
print(f"  - class_metrics_chart.png")
print(f"  - metrics_radar_chart.png")
print(f"  - sample_predictions.png")

print("\n" + "="*70)
print("TRAINING COMPLETED SUCCESSFULLY!")
print("="*70)